<a href="https://colab.research.google.com/github/Ikuesan/Projects/blob/main/Week_4_day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [3]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [30]:
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')
#anthropic_api_key = userdata.get('ANTHROPIC_API_KEY')   # optional
google_api_key = userdata.get('GOOGLE_API_KEY')         # optional
groq_api_key = userdata.get('GROQ_API_KEY')             # optional

openai = OpenAI(api_key=openai_api_key)
gemini = OpenAI(api_key=google_api_key, base_url = "https://generativelanguage.googleapis.com/v1beta/openai/")
groq = OpenAI(api_key=groq_api_key, base_url = "https://api.groq.com/openai/v1")

In [31]:
for m in groq.models.list().data:
    print(m.id)

openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
openai/gpt-oss-120b
openai/gpt-oss-20b


In [36]:
OPENAI_MODEL = "gpt-5"
#CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
#GROK_MODEL = "grok-4"
GEMINI_MODEL = "gemini-3.1-flash-lite"
GROQ_MODEL = "llama-prompt-guard-2-86m"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-haiku-4-5"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-3.1-flash-lite"

## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [26]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Linux',
  'arch': 'x86_64',
  'release': '6.6.159+',
  'version': '#1 SMP Thu Oct 22 03:07:54 UTC 2026',
  'kernel': '6.6.159+',
  'distro': {'name': 'Ubuntu 24.04.5 LTS', 'version': '24.04'},
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-linux-gnu'},
 'package_managers': ['apt'],
 'cpu': {'brand': 'Intel(R) Xeon(R) CPU @ 2.20GHz',
  'cores_logical': 2,
  'cores_physical': 1,
  'simd': ['AVX', 'AVX2', 'FMA', 'SSE4_2']},
 'toolchain': {'compilers': {'gcc': 'gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0',
   'g++': 'g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': 'cmake version 3.31.10',
   'ninja': '',
   'make': 'GNU Make 4.3'},
  'linkers': {'ld_lld': ''}}}

In [8]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))


You’re already set. Your system has g++ 13.3.0 installed, so you don’t need to install anything to compile and run a single C++ file.

For fastest runtime performance with g++, use high optimization, native CPU tuning, and LTO. In Python, set:

compile_command = ["g++", "-O3", "-march=native", "-flto", "-std=c++20", "main.cpp", "-o", "main"]
run_command = ["./main"]

Notes:
- -O3 enables aggressive optimizations.
- -march=native tunes the binary for your CPU (AVX2, etc.).
- -flto enables link-time optimization.
- -std=c++20 is broadly compatible; adjust if your code targets a different standard.

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [10]:
compile_command = ["g++", "-std=c++20", "-O3","-flto", "-march=native", "main.cpp", "-o", "main"]
run_command = ["./main"]

## And now, on with the main task

In [11]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [12]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]


In [13]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [14]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [15]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [16]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [17]:
run_python(pi)

Result: 3.141592656089
Execution Time: 55.747418 seconds


In [18]:
port(openai, OPENAI_MODEL, pi)

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [19]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [20]:
compile_and_run()

Result: 3.141592656089
Execution Time: 0.836266 seconds

Result: 3.141592656089
Execution Time: 0.852699 seconds

Result: 3.141592656089
Execution Time: 0.861793 seconds



In [ ]:
19.178207/0.082168

## OK let's try the other contenders!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [37]:
port(groq, GROQ_MODEL, pi)
compile_and_run()

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-prompt-guard-2-86m` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [29]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


Result: 3.141592656089
Execution Time: 0.826101 seconds

Result: 3.141592656089
Execution Time: 0.827637 seconds

Result: 3.141592656089
Execution Time: 0.851718 seconds



In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")